# Filosofi: Silver → Gold Layer Transformation
## Star Schema: Dimensions + Fact Tables

**Objective**: Transform filosofi silver layer into normalized fact tables for analysis

**Architecture**:
- **Dimensions**: Time, Geography
- **Facts** (grain: commune + year):
  - `fact_menages`: Household metrics
  - `fact_pauvrete`: Poverty rates & distribution
  - `fact_revenus`: Income composition breakdown
  - `fact_deciles`: Inequality metrics (deciles & ratios)

## Setup & Configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# Configuration des chemins
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SILVER_DIR = PROJECT_ROOT / "data" / "silver"
GOLD_DIR = PROJECT_ROOT / "data" / "gold" / "filosofi"

GOLD_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Dossier Silver : {SILVER_DIR}")
print(f"📁 Dossier Gold : {GOLD_DIR}")
print(f"\n⏰ Timestamp: {datetime.now().isoformat()}")

📁 Dossier Silver : /Users/zainfrayha/Desktop/electio-analytics-poc/data/silver
📁 Dossier Gold : /Users/zainfrayha/Desktop/electio-analytics-poc/data/gold/filosofi

⏰ Timestamp: 2026-04-01T19:19:34.824318


## Load Silver Layer Data

In [2]:
# Charger le fichier silver consolidé
silver_file = SILVER_DIR / "filosofi_2014_2021_commune_silver.csv"

print(f"Chargement: {silver_file.name}")
df_silver = pd.read_csv(silver_file, sep=";")

print(f"\n✅ Dimensions: {df_silver.shape[0]} rows × {df_silver.shape[1]} columns")
print(f"\n📊 Aperçu:")
display(df_silver.head())

print(f"\n📈 Valeurs uniques:")
print(f"  • Years: {sorted(df_silver['annee'].unique())}")
print(f"  • Communes: {df_silver['code_insee_commune'].nunique()}")
print(f"  • Records: {len(df_silver)}")

Chargement: filosofi_2014_2021_commune_silver.csv

✅ Dimensions: 2121 rows × 37 columns

📊 Aperçu:


,code_departement,code_commune,code_insee_commune,libelle_commune,annee,nb_menages_fiscaux,nb_personnes_menages_fiscaux,mediane_niveau_vie_euros,pct_menages_imposables,taux_pauvrete_ensemble,...,pct_impots,decile_1_revenu,decile_9_revenu,ratio_deciles,pct_revenus_non_salaries,source_dataset,date_refresh,source_extraction_url,date_ingestion,fichier_source
0,69,1,69001,Affoux,2014,124.0,361.5,21462.857143,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,filosofi_commune,2026-04-01 19:19:13,https://www.insee.fr/fr/statistiques/fichier/3...,2026-03-30T19:11:44.497047,INSEE_FILOSOFI_2014
1,69,2,69002,Aigueperse,2014,113.0,234.0,18494.500000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,filosofi_commune,2026-04-01 19:19:13,https://www.insee.fr/fr/statistiques/fichier/3...,2026-03-30T19:11:44.497047,INSEE_FILOSOFI_2014
2,69,3,69003,Albigny-sur-Saône,2014,958.0,2368.0,23454.761905,68.0,11.0,...,-19.7,12005.925926,43868.666667,3.653918,4.2,filosofi_commune,2026-04-01 19:19:13,https://www.insee.fr/fr/statistiques/fichier/3...,2026-03-30T19:11:44.497047,INSEE_FILOSOFI_2014
3,69,4,69004,Alix,2014,251.0,672.5,23311.914894,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,filosofi_commune,2026-04-01 19:19:13,https://www.insee.fr/fr/statistiques/fichier/3...,2026-03-30T19:11:44.497047,INSEE_FILOSOFI_2014
4,69,5,69005,Ambérieux,2014,200.0,583.5,22403.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,filosofi_commune,2026-04-01 19:19:13,https://www.insee.fr/fr/statistiques/fichier/3...,2026-03-30T19:11:44.497047,INSEE_FILOSOFI_2014



📈 Valeurs uniques:
  • Years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
  • Communes: 266
  • Records: 2121


## Schema Verification & Normalization

In [3]:
print("="*120)
print("📋 SCHEMA VERIFICATION: Silver Layer")
print("="*120 + "\n")

# Expected column groups
geographic_cols = {'code_departement', 'code_commune', 'code_insee_commune', 'libelle_commune'}
temporal_cols = {'annee'}
household_cols = {'nb_menages_fiscaux', 'nb_personnes_menages_fiscaux', 'mediane_niveau_vie_euros', 'pct_menages_imposables'}
poverty_cols = {'taux_pauvrete_ensemble', 'taux_pauvrete_moins_30ans', 'taux_pauvrete_30_39ans',
                 'taux_pauvrete_40_49ans', 'taux_pauvrete_50_59ans', 'taux_pauvrete_60_74ans',
                 'taux_pauvrete_75ans_plus', 'taux_pauvrete_proprietaires', 'taux_pauvrete_locataires'}
income_cols = {'pct_revenus_activite', 'pct_revenus_salaires_chomage', 'pct_revenus_activite_non_salariee',
               'pct_pensions_retraites', 'pct_revenus_patrimoine', 'pct_prestations_sociales',
               'pct_prestations_familiales', 'pct_minima_sociaux', 'pct_prestations_logement',
               'pct_impots', 'pct_revenus_non_salaries'}
decile_cols = {'decile_1_revenu', 'decile_9_revenu', 'ratio_deciles'}

all_cols = geographic_cols | temporal_cols | household_cols | poverty_cols | income_cols | decile_cols

actual_cols = set(df_silver.columns)
missing_cols = all_cols - actual_cols

print(f"✅ Geographic (4): {len(actual_cols & geographic_cols)}/{len(geographic_cols)}")
print(f"✅ Temporal (1): {len(actual_cols & temporal_cols)}/{len(temporal_cols)}")
print(f"✅ Household (4): {len(actual_cols & household_cols)}/{len(household_cols)}")
print(f"✅ Poverty (9): {len(actual_cols & poverty_cols)}/{len(poverty_cols)}")
print(f"✅ Income (11): {len(actual_cols & income_cols)}/{len(income_cols)}")
print(f"✅ Deciles (3): {len(actual_cols & decile_cols)}/{len(decile_cols)}")

print(f"\n{'✅ All expected columns present!' if not missing_cols else f'⚠️  Missing: {missing_cols}'}")
print("="*120 + "\n")

📋 SCHEMA VERIFICATION: Silver Layer

✅ Geographic (4): 4/4
✅ Temporal (1): 1/1
✅ Household (4): 4/4
✅ Poverty (9): 9/9
✅ Income (11): 11/11
✅ Deciles (3): 3/3

✅ All expected columns present!



In [4]:
# Normalisation et préparation des données
df_gold = df_silver.copy()

# Normaliser les codes INSEE (conserver les zéros significatifs)
df_gold['code_departement'] = df_gold['code_departement'].astype(str).str.strip().str.zfill(2)
df_gold['code_commune'] = df_gold['code_commune'].astype(str).str.strip().str.zfill(3)

# Créer id_commune au format INSEE standard (ex: 69001)
df_gold['id_commune'] = df_gold['code_departement'] + df_gold['code_commune']

# Créer id_year
df_gold['id_year'] = df_gold['annee'].astype(int)

print(f"✅ Données normalisées")
print(f"  • id_commune format: {df_gold['id_commune'].iloc[0]} (code_dept + code_comm)")
print(f"  • id_year values: {sorted(df_gold['id_year'].unique())}")
print(f"  • Total records: {len(df_gold)}")

✅ Données normalisées
  • id_commune format: 69001 (code_dept + code_comm)
  • id_year values: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
  • Total records: 2121


In [5]:
# Handle NaN values from INSEE statistical disclosure protection
# Note: Empty cells from CSV are automatically read as NaN by pandas

print("="*120)
print("🔍 NaN HANDLING: Statistical Disclosure Protection")
print("="*120 + "\n")

# Step 1: Verify NaN is properly recognized
print("Missing values per column (top 15 by sparsity):")
nan_counts = df_gold.isna().sum().sort_values(ascending=False)
print(nan_counts.head(15))

# Step 2: Explicitly convert any remaining empty strings to NaN (safety measure)
df_gold = df_gold.replace('', np.nan)

# Step 3: Convert numeric columns to proper float dtype (coerces non-numeric to NaN)
numeric_cols_list = [
    'nb_menages_fiscaux', 'nb_personnes_menages_fiscaux', 'mediane_niveau_vie_euros',
    'pct_menages_imposables', 'taux_pauvrete_ensemble', 'taux_pauvrete_moins_30ans',
    'taux_pauvrete_30_39ans', 'taux_pauvrete_40_49ans', 'taux_pauvrete_50_59ans',
    'taux_pauvrete_60_74ans', 'taux_pauvrete_75ans_plus', 'taux_pauvrete_proprietaires',
    'taux_pauvrete_locataires', 'pct_revenus_activite', 'pct_revenus_salaires_chomage',
    'pct_revenus_activite_non_salariee', 'pct_pensions_retraites', 'pct_revenus_patrimoine',
    'pct_prestations_sociales', 'pct_prestations_familiales', 'pct_minima_sociaux',
    'pct_prestations_logement', 'pct_impots', 'pct_revenus_non_salaries',
    'decile_1_revenu', 'decile_9_revenu', 'ratio_deciles'
]

for col in numeric_cols_list:
    df_gold[col] = pd.to_numeric(df_gold[col], errors='coerce')

print("\n✅ All numeric columns converted (empty strings → np.nan)")
print("\nData type verification:")
for col in numeric_cols_list[:5]:
    print(f"  • {col}: {df_gold[col].dtype}")

print("\n📊 Missing value summary:")
total_cells = df_gold.shape[0] * df_gold.shape[1]
total_nan = df_gold.isna().sum().sum()
nan_pct = (total_nan / total_cells * 100)
print(f"  • Total cells: {df_gold.shape[0]} rows × {df_gold.shape[1]} cols = {total_cells:,}")
print(f"  • Total NaN: {total_nan:,} ({nan_pct:.1f}%)")

🔍 NaN HANDLING: Statistical Disclosure Protection

Missing values per column (top 15 by sparsity):
taux_pauvrete_75ans_plus             2023
pct_revenus_salaires_chomage         2002
pct_revenus_activite                 2002
taux_pauvrete_moins_30ans            2000
taux_pauvrete_60_74ans               1983
taux_pauvrete_50_59ans               1919
taux_pauvrete_proprietaires          1886
taux_pauvrete_30_39ans               1875
taux_pauvrete_40_49ans               1838
taux_pauvrete_locataires             1676
taux_pauvrete_ensemble               1496
pct_revenus_activite_non_salariee    1269
pct_revenus_non_salaries             1150
pct_revenus_patrimoine               1150
pct_pensions_retraites               1150
dtype: int64

✅ All numeric columns converted (empty strings → np.nan)

Data type verification:
  • nb_menages_fiscaux: float64
  • nb_personnes_menages_fiscaux: float64
  • mediane_niveau_vie_euros: float64
  • pct_menages_imposables: float64
  • taux_pauvrete_ensemble:

## Dimension Tables

In [6]:
# --- DIM_TIME ---
# Unique years in dataset
dim_time = pd.DataFrame({
    'id_year': sorted(df_gold['id_year'].unique())
})

dim_time['annee'] = dim_time['id_year']
dim_time = dim_time[['id_year', 'annee']]

print(f"Dim_Time: {dim_time.shape[0]} years")
display(dim_time)

# Save
dim_time.to_csv(GOLD_DIR / "dim_time.csv", index=False, sep=";")
print(f"\n✅ Saved: dim_time.csv")

Dim_Time: 8 years


,id_year,annee
0,2014,2014
1,2015,2015
2,2016,2016
3,2017,2017
4,2018,2018
5,2019,2019
6,2020,2020
7,2021,2021



✅ Saved: dim_time.csv


In [7]:
# --- DIM_COMMUNE ---
# Geographic reference at commune level
cols_geo = ['id_commune', 'code_departement', 'code_commune', 'code_insee_commune', 'libelle_commune']
dim_commune = df_gold[cols_geo].drop_duplicates().sort_values('id_commune').reset_index(drop=True)

print(f"Dim_Commune: {dim_commune.shape[0]} communes")
display(dim_commune.head(10))

# Save
dim_commune.to_csv(GOLD_DIR / "dim_commune.csv", index=False, sep=";")
print(f"\n✅ Saved: dim_commune.csv")

Dim_Commune: 273 communes


,id_commune,code_departement,code_commune,code_insee_commune,libelle_commune
0,69001,69,001,69001,Affoux
1,69002,69,002,69002,Aigueperse
2,69003,69,003,69003,Albigny-sur-Saône
3,69004,69,004,69004,Alix
4,69005,69,005,69005,Ambérieux
5,69006,69,006,69006,Amplepuis
6,69007,69,007,69007,Ampuis
7,69008,69,008,69008,Ancy
8,69009,69,009,69009,Anse
9,69010,69,010,69010,L'Arbresle



✅ Saved: dim_commune.csv


## Fact Tables (Grain: id_commune + id_year)

In [8]:
# --- FACT_MENAGES ---
# Household & wealth metrics (grain: commune + year)
cols_menages = [
    'id_commune', 'id_year',
    'nb_menages_fiscaux',
    'nb_personnes_menages_fiscaux',
    'mediane_niveau_vie_euros',
    'pct_menages_imposables'
]

fact_menages = df_gold[cols_menages].drop_duplicates().sort_values(['id_commune', 'id_year']).reset_index(drop=True)

# Type conversion
fact_menages['nb_menages_fiscaux'] = pd.to_numeric(fact_menages['nb_menages_fiscaux'], errors='coerce')
fact_menages['nb_personnes_menages_fiscaux'] = pd.to_numeric(fact_menages['nb_personnes_menages_fiscaux'], errors='coerce')
fact_menages['mediane_niveau_vie_euros'] = pd.to_numeric(fact_menages['mediane_niveau_vie_euros'], errors='coerce')
fact_menages['pct_menages_imposables'] = pd.to_numeric(fact_menages['pct_menages_imposables'], errors='coerce')

print(f"Fact_Menages: {fact_menages.shape[0]} records")
print(f"  • Non-null households: {fact_menages['nb_menages_fiscaux'].notna().sum()}")
print(f"  • Non-null median income: {fact_menages['mediane_niveau_vie_euros'].notna().sum()}")
display(fact_menages.head())

# Save
fact_menages.to_csv(GOLD_DIR / "fact_menages.csv", index=False, sep=";")
print(f"\n✅ Saved: fact_menages.csv")

Fact_Menages: 2121 records
  • Non-null households: 2114
  • Non-null median income: 2114


,id_commune,id_year,nb_menages_fiscaux,nb_personnes_menages_fiscaux,mediane_niveau_vie_euros,pct_menages_imposables
0,69001,2014,124.0,361.5,21462.857143,NaN
1,69001,2015,132.0,375.0,21606.538462,NaN
2,69001,2016,133.0,382.5,21615.714286,NaN
3,69001,2017,136.0,383.0,22610.000000,NaN
4,69001,2018,145.0,399.0,22520.000000,NaN



✅ Saved: fact_menages.csv


In [9]:
# --- FACT_PAUVRETE ---
# Poverty metrics by age group and housing status (grain: commune + year)
cols_pauvrete = [
    'id_commune', 'id_year',
    'taux_pauvrete_ensemble',
    'taux_pauvrete_moins_30ans',
    'taux_pauvrete_30_39ans',
    'taux_pauvrete_40_49ans',
    'taux_pauvrete_50_59ans',
    'taux_pauvrete_60_74ans',
    'taux_pauvrete_75ans_plus',
    'taux_pauvrete_proprietaires',
    'taux_pauvrete_locataires'
]

fact_pauvrete = df_gold[cols_pauvrete].drop_duplicates().sort_values(['id_commune', 'id_year']).reset_index(drop=True)

# Type conversion
numeric_cols = cols_pauvrete[2:]  # All except id_commune, id_year
for col in numeric_cols:
    fact_pauvrete[col] = pd.to_numeric(fact_pauvrete[col], errors='coerce')

print(f"Fact_Pauvrete: {fact_pauvrete.shape[0]} records")
print(f"  • Non-null overall rate: {fact_pauvrete['taux_pauvrete_ensemble'].notna().sum()}")
print(f"  • Non-null <30y rate: {fact_pauvrete['taux_pauvrete_moins_30ans'].notna().sum()}")
print(f"  • Non-null elderly rate: {fact_pauvrete['taux_pauvrete_75ans_plus'].notna().sum()}")
display(fact_pauvrete.head())

# Save
fact_pauvrete.to_csv(GOLD_DIR / "fact_pauvrete.csv", index=False, sep=";")
print(f"\n✅ Saved: fact_pauvrete.csv")

Fact_Pauvrete: 2121 records
  • Non-null overall rate: 625
  • Non-null <30y rate: 121
  • Non-null elderly rate: 98


,id_commune,id_year,taux_pauvrete_ensemble,taux_pauvrete_moins_30ans,taux_pauvrete_30_39ans,taux_pauvrete_40_49ans,taux_pauvrete_50_59ans,taux_pauvrete_60_74ans,taux_pauvrete_75ans_plus,taux_pauvrete_proprietaires,taux_pauvrete_locataires
0,69001,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,69001,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,69001,2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,69001,2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,69001,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



✅ Saved: fact_pauvrete.csv


In [10]:
# --- FACT_REVENUS ---
# Income composition & transfers breakdown (grain: commune + year)
cols_revenus = [
    'id_commune', 'id_year',
    'pct_revenus_activite',
    'pct_revenus_salaires_chomage',
    'pct_revenus_activite_non_salariee',
    'pct_pensions_retraites',
    'pct_revenus_patrimoine',
    'pct_prestations_sociales',
    'pct_prestations_familiales',
    'pct_minima_sociaux',
    'pct_prestations_logement',
    'pct_impots',
    'pct_revenus_non_salaries'
]

fact_revenus = df_gold[cols_revenus].drop_duplicates().sort_values(['id_commune', 'id_year']).reset_index(drop=True)

# Type conversion
numeric_cols = cols_revenus[2:]  # All except id_commune, id_year
for col in numeric_cols:
    fact_revenus[col] = pd.to_numeric(fact_revenus[col], errors='coerce')

print(f"Fact_Revenus: {fact_revenus.shape[0]} records")
print(f"  • Non-null activity income: {fact_revenus['pct_revenus_activite'].notna().sum()}")
print(f"  • Non-null social benefits: {fact_revenus['pct_prestations_sociales'].notna().sum()}")
print(f"  • Non-null taxes: {fact_revenus['pct_impots'].notna().sum()}")
display(fact_revenus.head())

# Save
fact_revenus.to_csv(GOLD_DIR / "fact_revenus.csv", index=False, sep=";")
print(f"\n✅ Saved: fact_revenus.csv")

Fact_Revenus: 2121 records
  • Non-null activity income: 119
  • Non-null social benefits: 971
  • Non-null taxes: 971


,id_commune,id_year,pct_revenus_activite,pct_revenus_salaires_chomage,pct_revenus_activite_non_salariee,pct_pensions_retraites,pct_revenus_patrimoine,pct_prestations_sociales,pct_prestations_familiales,pct_minima_sociaux,pct_prestations_logement,pct_impots,pct_revenus_non_salaries
0,69001,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,69001,2015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,69001,2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,69001,2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,69001,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



✅ Saved: fact_revenus.csv


In [11]:
# --- FACT_DECILES ---
# Income inequality metrics (grain: commune + year)
cols_deciles = [
    'id_commune', 'id_year',
    'decile_1_revenu',
    'decile_9_revenu',
    'ratio_deciles'
]

fact_deciles = df_gold[cols_deciles].drop_duplicates().sort_values(['id_commune', 'id_year']).reset_index(drop=True)

# Type conversion
fact_deciles['decile_1_revenu'] = pd.to_numeric(fact_deciles['decile_1_revenu'], errors='coerce')
fact_deciles['decile_9_revenu'] = pd.to_numeric(fact_deciles['decile_9_revenu'], errors='coerce')
fact_deciles['ratio_deciles'] = pd.to_numeric(fact_deciles['ratio_deciles'], errors='coerce')

print(f"Fact_Deciles: {fact_deciles.shape[0]} records")
print(f"  • Non-null D1 (bottom 10%%): {fact_deciles['decile_1_revenu'].notna().sum()}")
print(f"  • Non-null D9 (top 10%%): {fact_deciles['decile_9_revenu'].notna().sum()}")
print(f"  • Non-null D9/D1 ratio: {fact_deciles['ratio_deciles'].notna().sum()}")
print(f"\n  • Ratio range: {fact_deciles['ratio_deciles'].min():.2f} to {fact_deciles['ratio_deciles'].max():.2f}")
display(fact_deciles.head())

# Save
fact_deciles.to_csv(GOLD_DIR / "fact_deciles.csv", index=False, sep=";")
print(f"\n✅ Saved: fact_deciles.csv")

Fact_Deciles: 2121 records
  • Non-null D1 (bottom 10%%): 971
  • Non-null D9 (top 10%%): 971
  • Non-null D9/D1 ratio: 971

  • Ratio range: 2.40 to 6.60


,id_commune,id_year,decile_1_revenu,decile_9_revenu,ratio_deciles
0,69001,2014,NaN,NaN,NaN
1,69001,2015,NaN,NaN,NaN
2,69001,2016,NaN,NaN,NaN
3,69001,2017,NaN,NaN,NaN
4,69001,2018,NaN,NaN,NaN



✅ Saved: fact_deciles.csv


## Summary & Quality Checks

In [12]:
print("\n" + "="*120)
print("✅ GOLD LAYER GENERATION COMPLETE")
print("="*120)

print(f"\n📊 DIMENSIONS:")
print(f"  • dim_time: {dim_time.shape[0]} years (2014-2021)")
print(f"  • dim_commune: {dim_commune.shape[0]} communes (Rhône dept)")

print(f"\n📈 FACT TABLES (Grain: commune + year = {fact_menages.shape[0]} base records):")
print(f"  • fact_menages: {fact_menages.shape[1]} columns (households, wealth)")
print(f"  • fact_pauvrete: {fact_pauvrete.shape[1]} columns (9 age/housing breakdowns)")
print(f"  • fact_revenus: {fact_revenus.shape[1]} columns (11 income components)")
print(f"  • fact_deciles: {fact_deciles.shape[1]} columns (inequality ratios)")

print(f"\n📁 Location: {GOLD_DIR}/")
print(f"\n📦 Files generated:")
for f in sorted(GOLD_DIR.glob("*.csv")):
    size = f.stat().st_size / 1024
    print(f"  ✓ {f.name} ({size:.1f} KB)")

print("\n" + "="*120)


✅ GOLD LAYER GENERATION COMPLETE

📊 DIMENSIONS:
  • dim_time: 8 years (2014-2021)
  • dim_commune: 273 communes (Rhône dept)

📈 FACT TABLES (Grain: commune + year = 2121 base records):
  • fact_menages: 6 columns (households, wealth)
  • fact_pauvrete: 11 columns (9 age/housing breakdowns)
  • fact_revenus: 13 columns (11 income components)
  • fact_deciles: 5 columns (inequality ratios)

📁 Location: /Users/zainfrayha/Desktop/electio-analytics-poc/data/gold/filosofi/

📦 Files generated:


  ✓ dim_commune.csv (8.8 KB)
  ✓ dim_time.csv (0.1 KB)
  ✓ fact_deciles.csv (53.1 KB)
  ✓ fact_menages.csv (76.7 KB)
  ✓ fact_pauvrete.csv (50.3 KB)
  ✓ fact_revenus.csv (76.4 KB)



In [13]:
# Quality checks
print("\n🔍 QUALITY ASSURANCE:")
print(f"\n✓ No duplicate (id_commune, id_year) pairs:")
print(f"  • fact_menages: {fact_menages.shape[0]} records, {fact_menages.duplicated(subset=['id_commune', 'id_year']).sum()} duplicates")
print(f"  • fact_pauvrete: {fact_pauvrete.shape[0]} records, {fact_pauvrete.duplicated(subset=['id_commune', 'id_year']).sum()} duplicates")
print(f"  • fact_revenus: {fact_revenus.shape[0]} records, {fact_revenus.duplicated(subset=['id_commune', 'id_year']).sum()} duplicates")
print(f"  • fact_deciles: {fact_deciles.shape[0]} records, {fact_deciles.duplicated(subset=['id_commune', 'id_year']).sum()} duplicates")

print(f"\n✓ Data distribution:")
print(f"  • Communes with complete poverty rates (both sexes): {fact_pauvrete[['taux_pauvrete_proprietaires', 'taux_pauvrete_locataires']].notna().all(axis=1).sum()}")
print(f"  • Communes with income data: {fact_revenus['pct_revenus_activite'].notna().sum()}")
print(f"  • Communes with decile ratios: {fact_deciles['ratio_deciles'].notna().sum()}")

print(f"\n✓ Geographic coverage per year:")
for year in sorted(df_gold['id_year'].unique()):
    communes_year = fact_menages[fact_menages['id_year'] == year]['id_commune'].nunique()
    print(f"  • {year}: {communes_year} communes")


🔍 QUALITY ASSURANCE:

✓ No duplicate (id_commune, id_year) pairs:
  • fact_menages: 2121 records, 0 duplicates


  • fact_pauvrete: 2121 records, 0 duplicates
  • fact_revenus: 2121 records, 0 duplicates
  • fact_deciles: 2121 records, 0 duplicates

✓ Data distribution:
  • Communes with complete poverty rates (both sexes): 227
  • Communes with income data: 119
  • Communes with decile ratios: 971

✓ Geographic coverage per year:
  • 2014: 266 communes
  • 2015: 266 communes
  • 2016: 265 communes
  • 2017: 265 communes
  • 2018: 265 communes
  • 2019: 264 communes
  • 2020: 265 communes
  • 2021: 265 communes
